# Demo 2 : Comparing Models and Prompts in the AI Playground

**Module 5A - Hour 2: Generative AI Fundamentals & the Databricks AI Platform (Topics 2.1-2.10)**

This notebook demonstrates Gen AI fundamentals using **Databricks SQL AI Functions** - the lowest-code entry point to generative AI on the platform.

Each section follows this pattern:
1. **Notes** (markdown) - concept explanation, the function signature, and *why* we're running this demo
2. **Demo** (SQL) - live code on sample data with inline comments

At the end you'll find a **Learning Conclusion** and a **Cleanup** cell that decommissions everything created during the demo.

In [0]:
%sql
-- ═══════════════════════════════════════════════════════════════
-- SETUP: Create catalog, schema, and sample data for all Gen AI demos
-- ═══════════════════════════════════════════════════════════════
-- We create a table of customer support tickets that we'll reuse
-- throughout the demo to showcase AI Functions on realistic data.
-- We create a dedicated catalog and schema for this demo, then a table
-- of customer support tickets that we'll reuse throughout the demo.
-- Using a dedicated catalog/schema makes cleanup easy and avoids
-- cluttering the default schema.

CREATE CATALOG IF NOT EXISTS module5a_demo2;
CREATE SCHEMA IF NOT EXISTS module5a_demo2.genai;

CREATE OR REPLACE TABLE module5a_demo2.genai.support_tickets (
  ticket_id   INT,
  customer    STRING,
  message     STRING,
  created_at  TIMESTAMP
);

INSERT INTO module5a_demo2.genai.support_tickets VALUES
(1,  'John Doe',      'I cannot log into my account, it keeps saying invalid credentials. Please help!',                       '2024-10-01 09:15:00'),
(2,  'Jane Smith',     'I was charged twice for my monthly subscription of $29.99. I need a refund immediately.',            '2024-10-01 10:30:00'),
(3,  'Bob Johnson',    'The app crashes every time I try to upload a photo. This is very frustrating!',                    '2024-10-01 11:45:00'),
(4,  'Alice Brown',    'I love the new dark mode feature! Great job team, keep it up.',                                     '2024-10-01 14:00:00'),
(5,  'Charlie Wilson', 'Can you add support for exporting data to Excel? That would be really helpful for my team.',    '2024-10-01 15:20:00'),
(6,  'Diana Prince',   'My order #12345 arrived damaged. The box was crushed and the product inside is broken.',        '2024-10-02 08:10:00'),
(7,  'Ethan Hunt',     'The new update is terrible. Everything is slower and the UI is confusing. I want the old version.','2024-10-02 09:30:00'),
(8,  'Fiona Green',   'I need to update my payment method from Visa ending 4521 to Mastercard ending 8830. Email: fiona@email.com', '2024-10-02 10:45:00'),
(9,  'George King',    '¿Cómo puedo cancelar mi suscripción? No encuentro la opción en la configuración.',                  '2024-10-02 12:00:00'),
(10, 'Hannah Lee',     'The API documentation is outdated. The endpoint /v2/users no longer exists and returns 404.',     '2024-10-02 13:15:00');

SELECT * FROM module5a_demo2.genai.support_tickets ORDER BY ticket_id;

ticket_id,customer,message,created_at
1,John Doe,"I cannot log into my account, it keeps saying invalid credentials. Please help!",2024-10-01T09:15:00.000Z
2,Jane Smith,I was charged twice for my monthly subscription of .99. I need a refund immediately.,2024-10-01T10:30:00.000Z
3,Bob Johnson,The app crashes every time I try to upload a photo. This is very frustrating!,2024-10-01T11:45:00.000Z
4,Alice Brown,"I love the new dark mode feature! Great job team, keep it up.",2024-10-01T14:00:00.000Z
5,Charlie Wilson,Can you add support for exporting data to Excel? That would be really helpful for my team.,2024-10-01T15:20:00.000Z
6,Diana Prince,My order #12345 arrived damaged. The box was crushed and the product inside is broken.,2024-10-02T08:10:00.000Z
7,Ethan Hunt,The new update is terrible. Everything is slower and the UI is confusing. I want the old version.,2024-10-02T09:30:00.000Z
8,Fiona Green,I need to update my payment method from Visa ending 4521 to Mastercard ending 8830. Email: fiona@email.com,2024-10-02T10:45:00.000Z
9,George King,¿Cómo puedo cancelar mi suscripción? No encuentro la opción en la configuración.,2024-10-02T12:00:00.000Z
10,Hannah Lee,The API documentation is outdated. The endpoint /v2/users no longer exists and returns 404.,2024-10-02T13:15:00.000Z


## 2.1-2.3 : Generative AI, Defined - What Gen AI Creates - Foundation Models vs LLMs

### Concepts
* **Generative AI** sits inside a nested field: AI → Machine Learning → Deep Learning → Generative AI.
  * *Predictive AI* forecasts outcomes (fraud detection, churn prediction).
  * *Generative AI* creates **new content**: text, code, image/audio/video, synthetic data, and structured (schema-valid) outputs.
* **Foundation models vs LLMs vs Gen AI** (nesting relationship):
  * Foundation models = large pre-trained models adaptable to many tasks.
  * LLMs = foundation models specialised for language.
  * Gen AI = the broadest category - includes image generators, music generators, code generators, etc.
* **Proprietary** (GPT, Claude, Gemini) vs **open-weight** (Llama, Gemma, Qwen): trade-offs in privacy, cost, and control.

`ai_gen(prompt)` is the **simplest Gen AI entry point** - it accepts a plain-text prompt and returns generated text. No endpoint name needed; Databricks picks the default model automatically. We use it to **generate a support-ticket taxonomy from scratch**, showing that Gen AI creates *structured text*, not just prose.

In [0]:
%sql
-- 2.1-2.3 Demo: Text generation with ai_gen()
-- ai_gen(prompt) → STRING  — sends a prompt to the default foundation model.
-- No endpoint name needed — Databricks selects the model for you.

SELECT ai_gen(
  'List exactly 5 short labels for categorising customer support tickets. '
  || 'Return only the labels, one per line, no numbering, no extra text.'
) AS generated_taxonomy;

generated_taxonomy
Technical Issue Payment Query Order Status Product Information Return Request


## 2.4 : How an LLM Turns a Prompt Into Output

### Concept
An LLM processes a prompt in four stages:
1. **Tokenize** - split text into sub-word tokens (the model's vocabulary).
2. **Embed** - convert each token into a vector that captures semantic meaning.
3. **Transformer (context)** - the self-attention mechanism lets every token “see” every other token and weigh its relevance.
4. **Predict** - output one token at a time; each prediction is fed back as input for the next step (autoregressive decoding).

`ai_query(endpoint, request, modelParameters => …)` lets us call a **specific** foundation model by name and control parameters such as `temperature` and `max_tokens`. By varying `temperature` (0 = deterministic, 1+ = creative) on the same prompt, we see how the probabilistic, autoregressive nature of token prediction produces different outputs.

> **Note:** If you get a 403 error mentioning Unity Gateway, replace `databricks-<model>` with `system.ai.<model>` in the endpoint name.

In [0]:
%sql
-- 2.4 Demo: Model parameters with ai_query()
-- ai_query(endpoint, request, modelParameters => named_struct(...)) lets us control:
--   temperature : 0.0 = deterministic, 1.0+ = creative
--   max_tokens  : caps the output length
-- We send the SAME prompt at two temperature settings to observe the difference.

SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'In one sentence, explain what Apache Spark is.',
  modelParameters => named_struct('temperature', 0.0, 'max_tokens', 50)
) AS response_temp_0;

SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'In one sentence, explain what Apache Spark is.',
  modelParameters => named_struct('temperature', 0.9, 'max_tokens', 50)
) AS response_temp_0_9;

response_temp_0_9
"Apache Spark is an open-source, unified analytics engine that provides high-performance, in-memory computing for processing large-scale data sets, supporting a wide range of workloads, including batch processing, interactive queries, and stream processing."


## 2.5-2.6 : Proprietary API vs Open-Weight Models - Choosing a Model

### Concepts
* **Proprietary APIs** (GPT, Claude, Gemini): vendor hosts the model; you call via API.
  * Strengths: state-of-the-art quality, no infrastructure to manage.
  * Trade-offs: data leaves your environment (egress cost, latency, governance gaps).
* **Open-weight models** (Llama, Gemma, Qwen): weights are downloadable; you host the model.
  * Strengths: full control, data stays in your environment, no per-token API cost.
  * Trade-offs: you manage infrastructure, quality may lag behind proprietary SOTA.
* **Four selection criteria**: **Privacy**, **Quality**, **Cost**, **Latency** - weigh these per use case.

We send the same prompt to two different models using `ai_query()` and compare the outputs side-by-side. On Databricks, both models are served through the **same unified API** - no separate SDKs or credentials to manage. This demonstrates the *quality* dimension of model selection.

> **Note:** If a model name returns an error, run `SHOW SERVING ENDPOINTS` to see which foundation model endpoints are available in your workspace.

In [0]:
%sql
-- 2.5-2.6 Demo: Compare two models on the same prompt
-- Model 1: Meta Llama 3.3 70B (open-weight, larger)
-- Model 2: Llama 4 Maverick (open-weight, newer architecture)
-- Same prompt, same temperature — compare quality and conciseness.

SELECT ai_query(
  'databricks-meta-llama-3-3-70b-instruct',
  'Classify this customer message into one word: billing, bug, feedback, shipping, account, or feature_request. Message: "I was charged twice for my subscription."',
  modelParameters => named_struct('temperature', 0.0, 'max_tokens', 10)
) AS llama_70b_response;

SELECT ai_query(
  'databricks-llama-4-maverick',
  'Classify this customer message into one word: billing, bug, feedback, shipping, account, or feature_request. Message: "I was charged twice for my subscription."',
  modelParameters => named_struct('temperature', 0.0, 'max_tokens', 50)
) AS llama_4_maverick_response;

llama_4_maverick_response
billing


## 2.7 : Why Success Is a Data Problem, Not a Model Problem

### Concept
* The model is increasingly a **commodity** - the differentiator is **your data**.
* **Data-native architecture**: keep data in your lakehouse, apply AI functions *in-place*.
  * No egress costs (exporting data to external APIs).
  * No latency from network round-trips.
  * No fragmented governance - everything stays under Unity Catalog.
* **Anti-pattern**: exporting data to an external LLM API, processing it, then re-importing results.

We run `ai_analyze_sentiment()` **directly on our governed table** - `module5a_demo2.genai.support_tickets`. The data never leaves Databricks. Unity Catalog controls who can read this table; the AI function is just another SQL operation on it. This is the data-native architecture in action.

In [0]:
%sql
-- 2.7 Demo: AI functions operate on governed data in-place
-- No data export, no external API calls — just SQL on a UC-governed table.
-- Unity Catalog controls who can read this table; the AI function respects those privileges.

SELECT 
  ticket_id,
  customer,
  ai_analyze_sentiment(message) AS sentiment
FROM module5a_demo2.genai.support_tickets
ORDER BY ticket_id;

ticket_id,customer,sentiment
1,John Doe,negative
2,Jane Smith,negative
3,Bob Johnson,negative
4,Alice Brown,positive
5,Charlie Wilson,positive
6,Diana Prince,negative
7,Ethan Hunt,negative
8,Fiona Green,neutral
9,George King,negative
10,Hannah Lee,negative


## 2.8 : AI Functions - The Lowest-Code Gen AI Entry Point

### AI Functions at a glance

| Function | What it does | Returns |
|---|---|---|
| `ai_analyze_sentiment(text)` | Positive / negative / neutral / mixed | STRING |
| `ai_classify(text, labels, MAP(...))` | Classify into custom labels | VARIANT |
| `ai_extract(text, schema, MAP(...))` | Pull structured fields from text | VARIANT |
| `ai_summarize(text, max_words)` | Condense text | STRING |
| `ai_translate(text, lang_code)` | Translate to target language | STRING |
| `ai_mask(text, array(...))` | Redact PII entities | STRING |
| `ai_gen(prompt)` | Free-form text generation | STRING |
| `ai_query(endpoint, request)` | Call a specific model endpoint | STRING/STRUCT |

We run each function on our `module5a_demo2.genai.support_tickets` table to show how AI Functions turn **unstructured customer messages** into **structured, actionable data** - all with a single line of SQL per function.

> **Important:** `ai_classify` and `ai_extract` should always use `MAP('version', '2.1')` to pin the recommended output contract.

In [0]:
%sql
-- 2.8a Demo: ai_analyze_sentiment() + ai_classify()
-- ai_analyze_sentiment(text) → 'positive' | 'negative' | 'neutral' | 'mixed'
-- ai_classify(text, labels_json, MAP(...)) → VARIANT with label + confidence
-- We classify each ticket into support categories with confidence scores.

SELECT
  ticket_id,
  message,
  ai_analyze_sentiment(message) AS sentiment,
  ai_classify(
    message,
    '{"billing":"Payment, invoice, or refund issues","bug":"App crashes, errors, broken features","feedback":"Praise, complaints, or general opinions","shipping":"Delivery or order damage","account":"Login, password, or profile issues","feature_request":"Suggestions for new features"}',
    MAP('version', '2.1', 'enableConfidenceScores', 'true')
  ) AS category_result
FROM module5a_demo2.genai.support_tickets
ORDER BY ticket_id;

ticket_id,message,sentiment,category_result
1,"I cannot log into my account, it keeps saying invalid credentials. Please help!",negative,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.78,""value"":""account""}]}"
2,I was charged twice for my monthly subscription of .99. I need a refund immediately.,negative,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.85,""value"":""billing""}]}"
3,The app crashes every time I try to upload a photo. This is very frustrating!,negative,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.85,""value"":""bug""}]}"
4,"I love the new dark mode feature! Great job team, keep it up.",positive,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.92,""value"":""feedback""}]}"
5,Can you add support for exporting data to Excel? That would be really helpful for my team.,positive,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.85,""value"":""feature_request""}]}"
6,My order #12345 arrived damaged. The box was crushed and the product inside is broken.,negative,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.85,""value"":""shipping""}]}"
7,The new update is terrible. Everything is slower and the UI is confusing. I want the old version.,negative,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.68,""value"":""feedback""}]}"
8,I need to update my payment method from Visa ending 4521 to Mastercard ending 8830. Email: fiona@email.com,neutral,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.78,""value"":""billing""}]}"
9,¿Cómo puedo cancelar mi suscripción? No encuentro la opción en la configuración.,negative,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.68,""value"":""account""}]}"
10,The API documentation is outdated. The endpoint /v2/users no longer exists and returns 404.,negative,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""confidence_score"":0.78,""value"":""bug""}]}"


In [0]:
%sql
-- 2.8b Demo: ai_extract()
-- ai_extract(text, schema_json, MAP(...)) → VARIANT
-- We extract structured fields: issue type, urgency level, and any mentioned amounts.
-- Schema uses the ai_extract user-schema format (NOT SQL DDL, NOT JSON Schema).
-- Allowed types: string, integer, number, boolean, enum, array, object.

SELECT
  ticket_id,
  message,
  ai_extract(
    message,
    '{
      "issue_type": {"type": "string", "description": "The type of customer issue"},
      "urgency": {"type": "enum", "labels": ["low", "medium", "high"], "description": "How urgent is this ticket"},
      "amount_mentioned": {"type": "number", "description": "Any dollar amount mentioned, null if none"}
    }',
    MAP('version', '2.1', 'enableConfidenceScores', 'true')
  ) AS extracted_fields
FROM module5a_demo2.genai.support_tickets
ORDER BY ticket_id;

ticket_id,message,extracted_fields
1,"I cannot log into my account, it keeps saying invalid credentials. Please help!","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""amount_mentioned"":{""confidence_score"":null,""value"":null},""issue_type"":{""confidence_score"":1,""value"":""Unable to log into my account""},""urgency"":{""confidence_score"":null,""value"":null}}}"
2,I was charged twice for my monthly subscription of .99. I need a refund immediately.,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""amount_mentioned"":{""confidence_score"":1,""value"":0.99},""issue_type"":{""confidence_score"":0.95,""value"":""Duplicate Subscription Charge""},""urgency"":{""confidence_score"":0.95,""value"":""high""}}}"
3,The app crashes every time I try to upload a photo. This is very frustrating!,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""amount_mentioned"":{""confidence_score"":null,""value"":null},""issue_type"":{""confidence_score"":1,""value"":""App crash during photo upload""},""urgency"":{""confidence_score"":1,""value"":""high""}}}"
4,"I love the new dark mode feature! Great job team, keep it up.","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""amount_mentioned"":{""confidence_score"":null,""value"":null},""issue_type"":{""confidence_score"":null,""value"":null},""urgency"":{""confidence_score"":null,""value"":null}}}"
5,Can you add support for exporting data to Excel? That would be really helpful for my team.,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""amount_mentioned"":{""confidence_score"":null,""value"":null},""issue_type"":{""confidence_score"":1,""value"":""feature request""},""urgency"":{""confidence_score"":null,""value"":null}}}"
6,My order #12345 arrived damaged. The box was crushed and the product inside is broken.,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""amount_mentioned"":{""confidence_score"":null,""value"":null},""issue_type"":{""confidence_score"":1,""value"":""damaged delivery""},""urgency"":{""confidence_score"":null,""value"":null}}}"
7,The new update is terrible. Everything is slower and the UI is confusing. I want the old version.,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""amount_mentioned"":{""confidence_score"":null,""value"":null},""issue_type"":{""confidence_score"":1,""value"":""software_update_complaint""},""urgency"":{""confidence_score"":null,""value"":null}}}"
8,I need to update my payment method from Visa ending 4521 to Mastercard ending 8830. Email: fiona@email.com,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""amount_mentioned"":{""confidence_score"":null,""value"":null},""issue_type"":{""confidence_score"":1,""value"":""payment method update""},""urgency"":{""confidence_score"":null,""value"":null}}}"
9,¿Cómo puedo cancelar mi suscripción? No encuentro la opción en la configuración.,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""amount_mentioned"":{""confidence_score"":null,""value"":null},""issue_type"":{""confidence_score"":1,""value"":""Subscription Cancellation""},""urgency"":{""confidence_score"":1,""value"":""low""}}}"
10,The API documentation is outdated. The endpoint /v2/users no longer exists and returns 404.,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":{""amount_mentioned"":{""confidence_score"":null,""value"":null},""issue_type"":{""confidence_score"":1,""value"":""outdated_documentation""},""urgency"":{""confidence_score"":null,""value"":null}}}"


In [0]:
%sql
-- 2.8c Demo: ai_summarize() + ai_translate()
-- ai_summarize(text, max_words) → STRING summary
-- ai_translate(text, lang_code) → STRING translated text
-- We summarize each ticket to 10 words and translate the Spanish ticket (#9) to English.

SELECT
  ticket_id,
  message,
  ai_summarize(message, 10) AS summary,
  CASE 
    WHEN ticket_id = 9 THEN ai_translate(message, 'en')
    ELSE NULL
  END AS translated_to_english
FROM module5a_demo2.genai.support_tickets
ORDER BY ticket_id;

ticket_id,message,summary,translated_to_english
1,"I cannot log into my account, it keeps saying invalid credentials. Please help!",Account login issue with invalid credentials,null
2,I was charged twice for my monthly subscription of .99. I need a refund immediately.,Refund needed for double charge,null
3,The app crashes every time I try to upload a photo. This is very frustrating!,App crashes when uploading photos,null
4,"I love the new dark mode feature! Great job team, keep it up.",Dark mode feature is well received,null
5,Can you add support for exporting data to Excel? That would be really helpful for my team.,Export data to Excel support needed,null
6,My order #12345 arrived damaged. The box was crushed and the product inside is broken.,Order arrived damaged and broken,null
7,The new update is terrible. Everything is slower and the UI is confusing. I want the old version.,Update is slow and confusing,null
8,I need to update my payment method from Visa ending 4521 to Mastercard ending 8830. Email: fiona@email.com,Update payment method to Mastercard,null
9,¿Cómo puedo cancelar mi suscripción? No encuentro la opción en la configuración.,Cancelar suscripción sin opción,How can I cancel my subscription? I don't find the option in the settings.
10,The API documentation is outdated. The endpoint /v2/users no longer exists and returns 404.,API documentation is outdated,null


In [0]:
%sql
-- 2.8d Demo: ai_mask()
-- ai_mask(text, array('entity_types')) → STRING with [MASKED] replacements
-- We redact emails, phone numbers, and credit card info from the tickets.
-- This is critical for compliance: masked data can be shared with wider teams.

SELECT
  ticket_id,
  customer,
  message AS original_message,
  ai_mask(message, array('email', 'phone', 'credit_card')) AS masked_message
FROM module5a_demo2.genai.support_tickets
ORDER BY ticket_id;

ticket_id,customer,original_message,masked_message
1,John Doe,"I cannot log into my account, it keeps saying invalid credentials. Please help!","I cannot log into my account, it keeps saying invalid credentials. Please help!"
2,Jane Smith,I was charged twice for my monthly subscription of .99. I need a refund immediately.,I was charged twice for my monthly subscription of .99. I need a refund immediately.
3,Bob Johnson,The app crashes every time I try to upload a photo. This is very frustrating!,The app crashes every time I try to upload a photo. This is very frustrating!
4,Alice Brown,"I love the new dark mode feature! Great job team, keep it up.","I love the new dark mode feature! Great job team, keep it up."
5,Charlie Wilson,Can you add support for exporting data to Excel? That would be really helpful for my team.,Can you add support for exporting data to Excel? That would be really helpful for my team.
6,Diana Prince,My order #12345 arrived damaged. The box was crushed and the product inside is broken.,My order #12345 arrived damaged. The box was crushed and the product inside is broken.
7,Ethan Hunt,The new update is terrible. Everything is slower and the UI is confusing. I want the old version.,The new update is terrible. Everything is slower and the UI is confusing. I want the old version.
8,Fiona Green,I need to update my payment method from Visa ending 4521 to Mastercard ending 8830. Email: fiona@email.com,I need to update my payment method from [MASKED] to [MASKED]. Email: [MASKED]
9,George King,¿Cómo puedo cancelar mi suscripción? No encuentro la opción en la configuración.,¿Cómo puedo cancelar mi suscripción? No encuentro la opción en la configuración.
10,Hannah Lee,The API documentation is outdated. The endpoint /v2/users no longer exists and returns 404.,The API documentation is outdated. The endpoint /v2/users no longer exists and returns 404.


## 2.9 : Compound AI Systems

### Concept
A compound AI system is not just a model - it combines:
* **Model** (LLM for understanding / generation)
* **Retrieval** (RAG for grounded knowledge)
* **Tools** (UC functions, code interpreters)
* **Classical ML** (predictive models)
* **Memory** (conversation context)
* **Guardrails** (safety filters, rate limits)

All components are governed by **Unity Catalog** - consistent access control, lineage, and audit.

We chain **three AI functions** into a single pipeline:
1. `ai_mask()` - redact PII from the message
2. `ai_classify()` - classify the cleaned message
3. `ai_summarize()` - produce a one-line summary for the support dashboard

This demonstrates **composability**: the output of one AI function feeds the input of the next, all in a single SQL query - no intermediate tables, no data movement.

In [0]:
%sql
-- 2.9 Demo: Compound AI pipeline — chain three AI functions in one query
-- Step 1: ai_mask()      → redact PII (emails, phones, credit cards)
-- Step 2: ai_classify()  → categorise the cleaned text into support categories
-- Step 3: ai_summarize()  → produce a dashboard-ready one-line summary
-- All in a single CTE + SELECT — no intermediate tables, no data movement.

WITH cleaned AS (
  SELECT
    ticket_id,
    customer,
    ai_mask(message, array('email', 'phone', 'credit_card')) AS masked_message
  FROM module5a_demo2.genai.support_tickets
)
SELECT
  ticket_id,
  customer,
  masked_message,
  ai_classify(
    masked_message,
    '{"billing":"Payment, invoice, or refund issues","bug":"App crashes, errors, broken features","feedback":"Praise, complaints, or general opinions","shipping":"Delivery or order damage","account":"Login, password, or profile issues","feature_request":"Suggestions for new features"}',
    MAP('version', '2.1')
  ) AS category,
  ai_summarize(masked_message, 8) AS dashboard_summary
FROM cleaned
ORDER BY ticket_id;

ticket_id,customer,masked_message,category,dashboard_summary
1,John Doe,"I cannot log into my account, it keeps saying invalid credentials. Please help!","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""value"":""account""}]}",Account login issue with credentials
2,Jane Smith,I was charged twice for my monthly subscription of .99. I need a refund immediately.,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""value"":""billing""}]}",Refund for double charge needed
3,Bob Johnson,The app crashes every time I try to upload a photo. This is very frustrating!,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""value"":""bug""}]}",App crashes when uploading photos
4,Alice Brown,"I love the new dark mode feature! Great job team, keep it up.","{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""value"":""feedback""}]}",Dark mode feature is great
5,Charlie Wilson,Can you add support for exporting data to Excel? That would be really helpful for my team.,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""value"":""feature_request""}]}",Export data to Excel support
6,Diana Prince,My order #12345 arrived damaged. The box was crushed and the product inside is broken.,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""value"":""shipping""}]}",Order arrived damaged and broken
7,Ethan Hunt,The new update is terrible. Everything is slower and the UI is confusing. I want the old version.,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""value"":""feedback""}]}",Update is slow and confusing
8,Fiona Green,I need to update my payment method from [MASKED] to [MASKED]. Email: [MASKED],"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""value"":""billing""}]}",Update payment method requested
9,George King,¿Cómo puedo cancelar mi suscripción? No encuentro la opción en la configuración.,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""value"":""account""}]}",Cancelar suscripción sin opción
10,Hannah Lee,The API documentation is outdated. The endpoint /v2/users no longer exists and returns 404.,"{""error_message"":null,""metadata"":{""version"":""2.1""},""response"":[{""value"":""bug""}]}",API documentation is outdated


## 2.10 : Databricks AI Platform - Five Workflow Stages

### The five stages
1. **Access a Model** - AI Playground, Foundation Model APIs, external models via AI Gateway
2. **Build** - AI Functions, Agent Framework, notebooks
3. **Prepare & Serve Data** - AI Search, Vector Search, Feature Store, Unity Catalog volumes
4. **Deploy** - Model Serving endpoints, Agent Bricks, Genie Spaces
5. **Govern & Monitor** - Unity Catalog, AI Gateway (guardrails, rate limits), Inference Tables

### Where this demo fits
Every function we ran today lives in **stage 2 (Build)** and operates on data in **stage 3 (Prepare & Serve Data)**. The tables are governed by **stage 5 (Govern & Monitor)**. We didn't need to set up any external infrastructure - everything ran inside a single SQL notebook.

In [0]:
# 2.10 Demo: The Deploy Stage - From AI Functions to Production
# This demo shows how the functions we ran connect to the Deploy stage.
# ai_query() calls a DEPLOYED model serving endpoint.
# In this demo, we used Foundation Model APIs (pre-deployed by Databricks).
# In production, you would deploy your own models to serving endpoints.

print("=== Deploy Stage: From AI Functions to Production ===")
print()

# 1. GOVERNED DATA (Stage 3: Prepare & Serve Data)
print("--- Stage 3: Prepare & Serve Data ---")
try:
    spark.sql("USE CATALOG module5a_demo2")
    tables = spark.sql("SHOW TABLES IN genai").collect()
    print(f"Governed tables in module5a_demo2.genai: {len(tables)}")
    for t in tables:
        print(f"  - module5a_demo2.genai.{t['tableName']}")
except Exception:
    print("  (Run the Setup cell first to create the catalog/schema)")
print()

# 2. DEPLOY (Stage 4: Deploy)
# The ai_query() calls we ran throughout this demo go through
# a deployed serving endpoint (Stage 4: Deploy).
print("--- Stage 4: Deploy ---")
print("Calling two deployed models on the SAME prompt...")
llama3 = spark.sql("""
  SELECT ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    'In one sentence, what does the Deploy stage mean on Databricks?',
    modelParameters => named_struct('temperature', 0.0, 'max_tokens', 80)
  ) AS response
""").collect()[0][0]

llama4 = spark.sql("""
  SELECT ai_query(
    'databricks-llama-4-maverick',
    'In one sentence, what does the Deploy stage mean on Databricks?',
    modelParameters => named_struct('temperature', 0.0, 'max_tokens', 80)
  ) AS response
""").collect()[0][0]

print(f"  Llama 3.3 70B: {llama3.strip()[:120]}")
print(f"  Llama 4 Maverick: {llama4.strip()[:120]}")
print()

# 3. THREE DEPLOYMENT OPTIONS
print("--- Three Deployment Options ---")
print("  a. Model Serving: deploy any model as a REST API endpoint (what ai_query calls)")
print("  b. Agent Bricks: managed agent platform (no-code deployment)")
print("  c. Genie Spaces: NL interface to governed data (zero-deployment)")
print()

# 4. GOVERN (Stage 5: Govern & Monitor)
print("--- Stage 5: Govern & Monitor ---")
catalogs = spark.sql("SHOW CATALOGS").collect()
print(f"Accessible catalogs: {len(catalogs)}")
for c in catalogs[:5]:
    print(f"  - {c['catalog']}")
if len(catalogs) > 5:
    print(f"  ... and {len(catalogs) - 5} more")
print()

print("=== Summary ===")
print("The 5 Databricks AI Platform stages:")
print("  1. Access a Model  -> AI Playground, Foundation Model APIs")
print("  2. Build           -> AI Functions, Agent Framework")
print("  3. Prepare & Serve  -> AI Search, Vector Search, UC tables")
print("  4. Deploy          -> Model Serving, Agent Bricks, Genie Spaces")
print("  5. Govern & Monitor-> Unity Catalog, AI Gateway, Inference Tables")

=== Deploy Stage: From AI Functions to Production ===

--- Stage 3: Prepare & Serve Data ---
  (Run the Setup cell first to create the catalog/schema)

--- Stage 4: Deploy ---
Calling two deployed models on the SAME prompt...
  Llama 3.3 70B: The Deploy stage on Databricks refers to the process of moving a tested and validated data engineering or data science a
  Llama 4 Maverick: The Deploy stage on Databricks refers to the process of taking validated data and models from the development or testing

--- Three Deployment Options ---
  a. Model Serving: deploy any model as a REST API endpoint (what ai_query calls)
  b. Agent Bricks: managed agent platform (no-code deployment)
  c. Genie Spaces: NL interface to governed data (zero-deployment)

--- Stage 5: Govern & Monitor ---
Accessible catalogs: 10
  - main
  - module5a_demo1
  - module5a_demo3
  - module5a_demo4
  - module5a_demo5
  ... and 5 more

=== Summary ===
The 5 Databricks AI Platform stages:
  1. Access a Model  -> AI Playgro

## Learning Conclusion

### What we demonstrated

| Topic | Function(s) Used | Key Takeaway |
|---|---|---|
| 2.1-2.3 | `ai_gen()` | Gen AI creates structured text from a prompt - the simplest entry point |
| 2.4 | `ai_query()` + `modelParameters` | Temperature controls determinism; LLMs predict one token at a time |
| 2.5-2.6 | `ai_query()` with two models | Model choice is a trade-off across privacy, quality, cost, latency |
| 2.7 | AI functions on a UC table | Data-native architecture: AI runs *on* your data, no export needed |
| 2.8 | `ai_analyze_sentiment`, `ai_classify`, `ai_extract`, `ai_summarize`, `ai_translate`, `ai_mask` | Six functions, one line of SQL each - unstructured text → structured data |
| 2.9 | Chained `ai_mask` → `ai_classify` → `ai_summarize` | Compound AI systems compose functions into pipelines |
| 2.10 | `ai_query()` on two models, `SHOW CATALOGS` | Deploy = model available as API endpoint; 5 platform stages governed by UC |

### Key principles
* **AI Functions are SQL** - no Python, no SDKs, no infrastructure. If you can write a SELECT, you can use Gen AI.
* **Data stays in the lakehouse** - no egress, no external API calls, full Unity Catalog governance.
* **Composability** - chain functions in a single query to build compound AI pipelines.
* **Model choice matters** - use `ai_gen` for convenience, `ai_query` for control.

In [0]:
%sql
-- ═══════════════════════════════════════════════════════════════
-- CLEANUP: Decommission everything created in this demo
-- ═══════════════════════════════════════════════════════════════
-- Drop the schema (cascades to tables) and catalog so no demo
-- artifacts remain in the workspace.

DROP SCHEMA IF EXISTS module5a_demo2.genai CASCADE;
DROP CATALOG IF EXISTS module5a_demo2 CASCADE;

-- Verify cleanup - catalog should no longer exist
SHOW CATALOGS LIKE 'module5a_demo2';

catalog
